# Bloch Sphere animation

## Imports

In [4]:
import matplotlib.pyplot as plt
import numpy as np
from qutip import *
from qutip.ipynbtools import plot_animation

%matplotlib inline

In [5]:
def qubit_integrate(w, theta, gamma1, gamma2, psi0, tlist):
    # Operators and the hamiltonian
    sx = sigmax()
    sy = sigmay()
    sz = sigmaz()
    sm = sigmam()
    H = w * (np.cos(theta) * sz + np.sin(theta) * sx)
    # collapse operators
    c_op_list = []
    n_th = 0.5  # temperature
    rate = gamma1 * (n_th + 1)
    if rate > 0.0:
        c_op_list.append(np.sqrt(rate) * sm)
    rate =  gamma1 * n_th
    if rate > 0.0:
        c_op_list.append(np.sqrt(rate) * sm.dag())
    rate = gamma2
    if rate > 0.0:
        c_op_list.append(np.sqrt(rate) * sz)

    # evolve and calculate expectation values
    output = mesolve(H, psi0, tlist, c_op_list, [sx, sy, sz])
    return output

In [7]:
w = 1.0 * 2 * np.pi # Qubit angular frequency
theta = 0.2 * np.pi # Qubit angle from sigma_z axis (toward sigma_x axis)
gamma1 = 0.5    # Qubit relaxation rate
gamma2 = 0.2    # Qubit dephasing rate
# initial state
a = 1.0
psi0 = (a * basis(2, 0) + (1 - a) * basis(2, 1)) / \
        (np.sqrt(a**2 + (1 - a )**2))
tlist = np.linspace(0, 4, 150)

In [8]:
result = qubit_integrate(w, theta, gamma1, gamma2, psi0, tlist)

/home/prem/VS Code/Learning/QuTip/.venv/lib/python3.13/site-packages/qutip/solver/solver_base.py:598: FutureWarning: e_ops will be keyword only from qutip 5.3 for all solver
  warnings.warn(


In [9]:
def plot_setup(result):

    fig = plt.figure(figsize=(8, 8))
    axes = fig.add_subplot(111, projection="3d", elev=30, azim=-40)

    return fig, axes

In [10]:
sphere = None


def plot_result(result, n, fig=None, axes=None):

    global sphere

    if fig is None or axes is None:
        fig, axes = plot_setup(result)

    if not sphere:
        sphere = Bloch(axes=axes)
        sphere.vector_color = ["r"]

    sphere.clear()
    sphere.add_vectors([result.expect[0][n],
                        result.expect[1][n],
                        result.expect[2][n]])
    sphere.add_points(
        [
            result.expect[0][: n + 1],
            result.expect[1][: n + 1],
            result.expect[2][: n + 1],
        ],
        meth="l",
    )
    sphere.make_sphere()

    return axes.artists

In [11]:
# You can choose your own writer and codec here.
# Setting codec=None sets the codec to the standard
# defined in matplotlib.rcParams['animation.codec']
plot_animation(plot_setup, plot_result, result, writer="ffmpeg", codec=None)

<Figure size 500x500 with 0 Axes>